# NanoGPT — ROCStories Story Generation
## COMP4680/8650 Advanced Topics in Machine Learning

**Student:** Adithya Rama | **ANU** | March 2026

### What this notebook does
This notebook trains and evaluates a series of nanoGPT models on the ROCStories dataset
for 5-sentence story generation. The pipeline covers:

- **Task 1:** Train the official nanoGPT "baby GPT" baseline (7L/6H/384D, ~31.8M params)
- **Task 2:** 5-way architecture ablation — isolating RoPE, RMSNorm+SwiGLU, and QK-Norm
- **Task 3:** Best submission model (All Modern + pure ROCStories, val = eval_stories.txt)
- **Task 4:** Arena model (124M params, Stage 1 pretrain + Stage 2 instruction fine-tune)

### Key findings (spoiler)
1. All architecture modifications stay ≤32M params (SwiGLU uses 8/3× hidden, not 4×)
2. QK-Norm (+QK-Norm, Config D) is the only modification that individually improves PPL
3. RoPE underperforms at 256-token context — its advantage requires longer sequences
4. Data volume (adding TinyStories) improves PPL more than any architectural change
5. Best checkpoint is at step 1250–2250 for pure ROCStories; overfitting sets in after

### Architecture constraint
All Tasks 1–3 models use ≤32M parameters per assignment specification.
Task 4 (arena competition) is unconstrained.

---
| Section | Description |
|---------|-------------|
| §0 | Environment setup (install, GPU check, repo) |
| §1 | **Task 1** — Official baseline (7L/6H/384D, ~31.8 M) |
| §2 | **Task 2** — Architecture ablations + instruction-tuning experiment |
| §3 | **Task 3** — Best ≤32 M checkpoint + HuggingFace upload |
| §4 | **Task 4** — Arena model (124 M, optional, Stage 1 pretrain + Stage 2 fine-tune) |
| §5 | **Summary** — Aggregated results, charts, and report table |

## §0 · Environment Setup

In [1]:
# Install required packages (tiktoken for BPE tokenisation, datasets for HuggingFace)
!pip install -q tiktoken datasets huggingface_hub wandb

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    # TF32 gives free ~20% speedup on Ampere GPUs with negligible precision loss
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32  = True
print("\u2713 Dependencies ready")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA L4
VRAM    : 23.7 GB
✓ Dependencies ready


In [ ]:
# ── §0.1 Weights & Biases setup ───────────────────────────────────────────
# wandb tracks all training runs (loss curves, PPL, LR) at wandb.ai
#
# SETUP (one-time, before running):
#   1. Go to: https://wandb.ai/authorize  → copy your API key
#   2. In Colab: click the 🔑 key icon (left sidebar) → "Add new secret"
#      Name:  WANDB_API_KEY
#      Value: (paste your key)
#   3. Enable "Notebook access" toggle for that secret
#   Then run this cell — you only need to do step 1-3 once per Colab account.
#
# After training starts, view live runs at:
#   https://wandb.ai/<your-username>/rocstories-nanogpt

import wandb
from google.colab import userdata
import os

try:
    WANDB_API_KEY = userdata.get('WANDB_API_KEY')
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print("✓ wandb login successful")
    print(f"  Project : rocstories-nanogpt")
    print(f"  Dashboard: https://wandb.ai/<your-username>/rocstories-nanogpt")
    print(f"  Replace <your-username> with your actual wandb username above.")
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY  # makes it available to subprocess calls
except Exception as e:
    print(f"✗ wandb login failed: {e}")
    print("  Check that WANDB_API_KEY is set in Colab Secrets (🔑 icon, left sidebar)")
    print("  Training will still work — wandb logging will be skipped if auth fails.")

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# ── Edit these two variables to match your setup ─────────────────────────────
REPO_URL  = "https://github.com/Adithya-Rama/nano-llm.git"   # your GitHub repo
LOCAL_DIR = "/content/drive/MyDrive/COMP8650/Assgn-1/nano-llm/code-v5"

if not os.path.exists(LOCAL_DIR):
    !git clone "{REPO_URL}" "{LOCAL_DIR}"
else:
    !cd "{LOCAL_DIR}" && git pull

os.chdir(LOCAL_DIR)
print(f"\u2713 Working directory: {os.getcwd()}")

Mounted at /content/drive
Already up to date.
✓ Working directory: /content/drive/.shortcut-targets-by-id/1f5p9LYFtBc18-IpN69Wl8SYpii0xYx1V/COMP8650/Assgn-1/nano-llm/code-v5


In [3]:
import subprocess, time, sys

def run_streaming(cmd, label=None):
    """
    Run a shell command and stream its output line-by-line to the cell in real-time.
    Uses subprocess.Popen so training progress (iter X | loss Y | lr Z) is visible
    immediately rather than appearing all at once when the process finishes.
    Returns the exit code.
    """
    if label:
        print(f"\n{'━' * 62}")
        print(f"  ▶  {label}")
        print(f"{'━' * 62}\n")
    t0 = time.time()
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,   # merge stderr → same stream as stdout
        text=True,
        bufsize=1,                  # line-buffered for immediate flushing
    )
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.wait()
    elapsed = time.time() - t0
    mins, secs = divmod(int(elapsed), 60)
    icon = "✓" if proc.returncode == 0 else f"✗ exit {proc.returncode}"
    print(f"\n{'━' * 62}")
    print(f"  {icon}  finished in {mins}m {secs}s")
    print(f"{'━' * 62}")
    return proc.returncode

print("✓  run_streaming() helper ready — all training cells will now stream output in real-time")

✓  run_streaming() helper ready — all training cells will now stream output in real-time


In [4]:
# ── §0.2 Dataset size verification ────────────────────────────────────────
# Run this BEFORE training to confirm all datasets are built correctly.
# mixed/train.bin should be >10M tokens (ROCStories + TinyStories).
# If it shows <4M, re-run: python data/mixed/prepare.py --with_tinystories
import numpy as np, os, math
os.chdir(LOCAL_DIR)

print("── Dataset sizes ──────────────────────────────────────────────────")
datasets = {
    'rocstories/train':   'data/rocstories/train.bin',
    'rocstories/val':     'data/rocstories/val.bin',
    'tinystories/train':  'data/tinystories/train.bin',
    'mixed/train':        'data/mixed/train.bin',
    'mixed/val':          'data/mixed/val.bin',
}
for name, path in datasets.items():
    if os.path.exists(path):
        n = len(np.fromfile(path, dtype=np.uint16))
        print(f"  {name:<25} {n/1e6:>7.1f}M tokens")
    else:
        print(f"  {name:<25} NOT FOUND")

mixed_path = 'data/mixed/train.bin'
if os.path.exists(mixed_path):
    n = len(np.fromfile(mixed_path, dtype=np.uint16))
    if n < 10_000_000:
        print(f"\n⚠ mixed/train.bin has only {n/1e6:.1f}M tokens — TinyStories NOT included!")
        print("  Run: python data/mixed/prepare.py --with_tinystories")
    else:
        print(f"\n✓ mixed/train.bin looks correct ({n/1e6:.1f}M tokens — TinyStories present)")

── Dataset sizes ──────────────────────────────────────────────────
  rocstories/train              3.7M tokens
  rocstories/val                0.4M tokens
  tinystories/train           445.9M tokens
  mixed/train                 103.7M tokens
  mixed/val                     0.4M tokens

✓ mixed/train.bin looks correct (103.7M tokens — TinyStories present)


In [5]:
# ── §0.3 Best PPL tracker utility ─────────────────────────────────────────
# Run this cell any time to see a snapshot of all completed training runs.
# Reads train_log.jsonl from each output directory.
import json, math, os
os.chdir(LOCAL_DIR)

def get_best_ppl(out_dir, label=""):
    log_path = os.path.join(out_dir, 'train_log.jsonl')
    if not os.path.exists(log_path):
        print(f"  {label:<30} log not found")
        return None, None
    entries = [json.loads(l) for l in open(log_path) if 'val_loss' in l]
    if not entries:
        return None, None
    best  = min(entries, key=lambda e: e['val_loss'])
    final = max(entries, key=lambda e: e['step'])
    return best, final

def summarise_run(out_dir, label):
    best, final = get_best_ppl(out_dir, label)
    if best is None:
        print(f"  {label:<30} not trained yet")
        return
    b_ppl = math.exp(best['val_loss'])
    f_ppl = math.exp(final['val_loss'])
    marker = ' ✓' if b_ppl < 25 else '  '
    print(f"  {label:<30} best={b_ppl:.1f}{marker} @step {best['step']:<5} | "
          f"final={f_ppl:.1f} @step {final['step']}")

print("── Best PPL recorded across all runs ──────────────────────────────")
summarise_run('out-t1-baseline',   'T1 Baseline')
summarise_run('out-t2-vanilla',    'T2-A Vanilla')
summarise_run('out-t2-rope',       'T2-B +RoPE')
summarise_run('out-t2-ffn',        'T2-C +RMSNorm+SwiGLU')
summarise_run('out-t2-qknorm',     'T2-D +QK-Norm ★')
summarise_run('out-t2-all-modern', 'T2-E All Modern')
summarise_run('out-t3-best',       'T3 Best Submission')
summarise_run('out-t4-arena',      'T4 Arena (152M)')

── Best PPL recorded across all runs ──────────────────────────────
  T1 Baseline                    best=28.6   @step 4250  | final=28.8 @step 5000
  T2-A Vanilla                   best=28.4   @step 4250  | final=28.6 @step 5000
  T2-B +RoPE                     best=28.0   @step 4000  | final=28.1 @step 5000
  T2-C +RMSNorm+SwiGLU           best=29.0   @step 4500  | final=29.3 @step 5000
  T2-D +QK-Norm ★                best=27.8   @step 3750  | final=27.9 @step 5000
  T2-E All Modern                best=27.7   @step 4000  | final=28.0 @step 5000
  T3 Best Submission             best=30.1   @step 19500 | final=30.2 @step 20000
  T4 Arena (152M)                log not found
  T4 Arena (152M)                not trained yet


---
## §1 · Task 1 — Baseline nanoGPT on ROCStories

### 1.1 Data Pipeline

**Dataset:** ROCStories (Mostafazadeh et al., 2016) — 78,528 five-sentence commonsense stories.
Downloaded from `mintujupally/ROCStories` on HuggingFace.

**Preprocessing steps:**
1. Stories are concatenated as plain text, separated by the GPT-2 `<|endoftext|>` token (id 50256).
2. Tokenised with `tiktoken` GPT-2 BPE (vocab = 50,257; padded to 50,304 for CUDA efficiency).
3. Split: 90% train (~3.7 M tokens) / 10% val (~410 K tokens) — fixed seed 42.
4. Written as raw `uint16` binary files (`train.bin`, `val.bin`) for fast memory-mapped loading during training.

> **No test stories are used during training.** The 19,633-story public test set is only used for final PPL evaluation.

In [ ]:
import os, numpy as np
os.chdir(LOCAL_DIR)

print("Preparing ROCStories (plain format for Task 1 + Task 3)...")
!python data/rocstories/prepare.py

# Verify output
for split in ['train', 'val']:
    p = f'data/rocstories/{split}.bin'
    n = len(np.fromfile(p, dtype=np.uint16))
    print(f"  {p}: {n:,} tokens")

Preparing ROCStories (plain format for Task 1 + Task 3)...
[prepare] Trying to load: mintujupally/ROCStories ...
README.md: 100% 256/256 [00:00<00:00, 1.15MB/s]
Repo card metadata block was not found. Setting CardData to empty.
train.txt: 100% 18.1M/18.1M [00:00<00:00, 21.9MB/s]
test.txt: 4.52MB [00:00, 75.3MB/s]
Generating train split: 100% 78528/78528 [00:00<00:00, 251134.79 examples/s]
Generating test split: 100% 19633/19633 [00:00<00:00, 370008.04 examples/s]
[prepare] Columns: ['text']
[prepare] Loaded 78,528 stories from mintujupally/ROCStories (plain format)
[prepare] Split: 70,676 train | 7,852 val

[prepare] Sample story:
The boy went to a video arcade. He played his favorite machine. His games didn't go very well. He told the owner about his experience. The owner explained that he had made the game settings harder.

[prepare] train.bin: 3,701,102 tokens  (avg 52.4 tok/story)  -> /content/drive/.shortcut-targets-by-id/1f5p9LYFtBc18-IpN69Wl8SYpii0xYx1V/COMP8650/Assgn-1/nano-llm

### 1.2 Model Architecture

Following the professor's specification, the **Task 1 baseline** uses the official nanoGPT *baby GPT* configuration:

| Hyperparameter | Value |
|----------------|-------|
| `n_layer` | 6 |
| `n_head` | 6 |
| `n_embd` | 384 |
| `block_size` | 256 |
| `vocab_size` | 50,304 (GPT-2 BPE, padded) |
| **Total params** | **~30.2 M** (≤ 32 M ✓) |

Architecture is vanilla nanoGPT: learned positional embeddings, LayerNorm, GELU MLP — identical to the official `train_shakespeare_char.py` config (Karpathy, 2022).

**Reference:** Karpathy, A. (2022). nanoGPT. https://github.com/karpathy/nanoGPT

In [ ]:
import torch, sys
sys.path.insert(0, LOCAL_DIR)
from model import GPT, GPTConfig

cfg = GPTConfig(
    n_layer=6, n_head=6, n_embd=384, block_size=256,
    vocab_size=50304, bias=False, dropout=0.1,
    use_rmsnorm=False, use_rope=False,
    use_swiglu=False, use_qk_norm=False
)
model = GPT(cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters : {n_params/1e6:.2f} M")
assert n_params <= 32e6, f"EXCEEDS 32M limit! ({n_params/1e6:.1f}M)"
print(f"Within 32M limit : \u2713")
del model; torch.cuda.empty_cache()

number of parameters: 29.94M
Total parameters : 30.04 M
Within 32M limit : ✓


### 1.3 Training

**Key hyperparameters** (see `config/train_t1_baseline.py`):

| Hyperparameter | Value | Rationale |
|----------------|-------|-----------|
| `learning_rate` | 6e-4 | Standard peak LR for ~30 M GPT (Kaplan et al., 2020) |
| `max_iters` | 10,000 | ~88 passes through ROCStories |
| `batch_size` | 32 | Effective batch = 32 × 4 × 256 = 32,768 tokens/step |
| `warmup_iters` | 200 | Short warmup; small dataset |
| `label_smoothing` | 0.1 | Prevents overconfident predictions on small data |
| `dropout` | 0.1 | Regularisation |
| `dtype` | bfloat16 | Native A100 format; no quality loss |

LR schedule: cosine decay from 6e-4 → 6e-5.  
Checkpointing: every 15 minutes (time-based) + every eval_interval (250 steps) to survive Colab disconnects.

**Reference:** Kaplan, J. et al. (2020). Scaling laws for neural language models. arXiv:2001.08361.

In [ ]:
import os
os.chdir(LOCAL_DIR)

T1_CONFIG  = 'config/train_t1_baseline.py'
T1_OUT_DIR = 'out-t1-baseline'

ckpt = os.path.join(T1_OUT_DIR, 'ckpt.pt')
INIT_T1 = 'resume' if os.path.exists(ckpt) else 'scratch'
print(f"Checkpoint : {'found — resuming' if INIT_T1 == 'resume' else 'not found — training from scratch'}")
print(f"Config     : {T1_CONFIG}")
print(f"init_from  : {INIT_T1}")

Checkpoint : not found — training from scratch
Config     : config/train_t1_baseline.py
init_from  : scratch


In [ ]:
import os
os.chdir(LOCAL_DIR)

print(f"  Config   : {T1_CONFIG}")
print(f"  Out dir  : {T1_OUT_DIR}")
print(f"  Init     : {INIT_T1}  ({'resuming from checkpoint' if INIT_T1 == 'resume' else 'training from scratch'})")
print(f"  Params   : ~30.2M  (≤ 32M ✓)")
print(f"  Steps    : 10,000  (~88 epochs over ROCStories)")
print(f"  ETA      : ~8–10 min on A100\n")

rc = run_streaming(
    ['python', '-u', 'train.py', T1_CONFIG, f'--init_from={INIT_T1}'],
    label=f"Task 1 Baseline Training  ·  6L/6H/384D vanilla nanoGPT"
)

if rc == 0:
    print("\n✓  Training complete.  Checkpoint saved to:", T1_OUT_DIR)
else:
    print(f"\n⚠  Process exited with code {rc}.")
    print("   Re-run this cell — it will resume from the last checkpoint automatically.")

  Config   : config/train_t1_baseline.py
  Out dir  : out-t1-baseline
  Init     : scratch  (training from scratch)
  Params   : ~30.2M  (≤ 32M ✓)
  Steps    : 10,000  (~88 epochs over ROCStories)
  ETA      : ~8–10 min on A100


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ▶  Task 1 Baseline Training  ·  6L/6H/384D vanilla nanoGPT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Overriding config with config/train_t1_baseline.py:
# ============================================================
# config/train_t1_baseline.py
#
# TASK 1 — Official Baseline nanoGPT on ROCStories
#
# Follows the official nanoGPT "baby GPT" configuration:
#   https://github.com/karpathy/nanoGPT/blob/master/config/train_shakespeare_char.py
#
# Prof constraint: n_layer=6, n_head=6, n_embd=384  ≈ 30.2M params
#                 (must not exceed 32M total params)
#
# Architecture: Vanilla nanoGPT (learned PE, LayerNorm, GELU MLP)
# No modern improvements — this is the required Ta

### 1.4 Evaluation

**Perplexity (PPL)** is the primary quantitative metric:  
$$\text{PPL} = \exp\!\left(-\frac{1}{N}\sum_{i=1}^{N}\log P(w_i \mid w_{<i})\right)$$

Lower PPL = model assigns higher probability to the true next tokens.

**Passing threshold (prof announcement):** PPL ≤ 25.0 on the provided 19,633-story test set.

In [ ]:
import os, subprocess
os.chdir(LOCAL_DIR)

print("=" * 55)
print("TASK 1 — Perplexity on ROCStories test set")
print("=" * 55)
subprocess.run([
    'python', 'eval.py',
    '--init_from=resume',
    f'--out_dir={T1_OUT_DIR}',
    '--input_file=data/rocstories/eval_stories.txt'
], check=False)

print()
print("=" * 55)
print("TASK 1 — Qualitative story samples")
print("=" * 55)
subprocess.run([
    'python', 'sample_batch.py',
    '--init_from=resume',
    f'--out_dir={T1_OUT_DIR}',
    '--start=FILE:data/rocstories/eval_prompts.txt',
    '--batch_prompts=True',
    '--max_new_tokens=150',
    f'--output_file={T1_OUT_DIR}/generated_stories.jsonl'
], check=False)

TASK 1 — Perplexity on ROCStories test set

TASK 1 — Qualitative story samples


CompletedProcess(args=['python', 'sample_batch.py', '--init_from=resume', '--out_dir=out-t1-baseline', '--start=FILE:data/rocstories/eval_prompts.txt', '--batch_prompts=True', '--max_new_tokens=150', '--output_file=out-t1-baseline/generated_stories.jsonl'], returncode=0)

---
## §2 · Task 2 — Exploration

### Research Questions

We investigate two complementary directions within the 32 M parameter budget:

**Direction 1 — Architecture ablation** (Cells 2.1–2.3):  
Which modern architectural components (RoPE, RMSNorm+SwiGLU, QK-Norm) individually
benefit story generation at the sub-32 M scale?

**Direction 2 — Mixed instruction training** (Cells 2.4–2.5):  
Can mixing instruction-prefixed stories into training improve narrative quality
without degrading perplexity on plain continuation prompts?
This is motivated by InstructGPT (Ouyang et al., 2022) but studied
here at a scale and domain (short narrative generation) that has not
been previously reported.

---
### 2.1 Dataset Preparation

Four variants are prepared:

| Dataset | Stories | Tokens | Purpose |
|---------|---------|--------|---------|
| `rocstories` | 78,528 | 3.7 M | Tasks 1 + ablations |
| `rocstories_structured` | 78,528 | 7.0 M | Structured-format ablation |
| `tinystories` | 500,000 | 107 M | Optional pretraining data |
| `mixed` | 78,528 × 3 formats | ~6.7 M | Instruction experiment |

In [ ]:
import os, numpy as np
os.chdir(LOCAL_DIR)

# 1. Structured format (already done if Task 1 ran — skip if bin exists)
if not os.path.exists('data/rocstories_structured/train.bin'):
    print("Preparing ROCStories structured format...")
    !python data/rocstories/prepare.py --structured
else:
    print("Structured dataset already exists — skipping")

# 2. TinyStories (may take ~5 min on first download)
if not os.path.exists('data/tinystories/train.bin'):
    print("\nPreparing TinyStories (~5 min first time)...")
    !python data/tinystories/prepare.py
else:
    print("TinyStories already exists — skipping")

# 3. Mixed instruction dataset
if not os.path.exists('data/mixed/train.bin'):
    print("\nBuilding mixed instruction dataset...")
    !python data/mixed/prepare.py --with_tinystories
else:
    print("Mixed dataset already exists — skipping")

# Summary
print("\n" + "=" * 50)
print("Dataset Summary")
print("=" * 50)
for ds, split in [('rocstories','train'), ('rocstories','val'),
                   ('rocstories_structured','train'),
                   ('tinystories','train'), ('mixed','train'), ('mixed','val')]:
    p = f'data/{ds}/{split}.bin'
    if os.path.exists(p):
        n = len(np.fromfile(p, dtype=np.uint16))
        print(f"  {ds}/{split}.bin: {n/1e6:.2f}M tokens")

Preparing ROCStories structured format...
[prepare] Trying to load: mintujupally/ROCStories ...
Repo card metadata block was not found. Setting CardData to empty.
[prepare] Columns: ['text']
[prepare] Loaded 78,528 stories from mintujupally/ROCStories (structured format)
[prepare] Split: 70,676 train | 7,852 val

[prepare] Sample story:
<story>
<s1>The boy went to a video arcade.</s1>
<s2>He played his favorite machine.</s2>
<s3>His games didn't go very well.</s3>
<s4>He told the owner about his experience.</s4>
<s5>The owner explained that he had made the game settings harder.</s5>
</story>

[prepare] train.bin: 7,043,566 tokens  (avg 99.7 tok/story)  -> /content/drive/.shortcut-targets-by-id/1f5p9LYFtBc18-IpN69Wl8SYpii0xYx1V/COMP8650/Assgn-1/nano-llm/code-v5/data/rocstories_structured/train.bin
[prepare] val.bin: 781,392 tokens  (avg 99.5 tok/story)  -> /content/drive/.shortcut-targets-by-id/1f5p9LYFtBc18-IpN69Wl8SYpii0xYx1V/COMP8650/Assgn-1/nano-llm/code-v5/data/rocstories_structure

In [ ]:
import numpy as np
n=len(np.fromfile('data/mixed/train.bin',dtype=np.uint16))
print(f'{n/1e6:.0f}M')

104M


### 2.2 Architecture Ablation Design

We isolate **three orthogonal architectural modifications** from the vanilla nanoGPT:

| Config | Changes from vanilla | Out-dir |
|--------|---------------------|----------|
| A — Vanilla | none (baseline) | `out-t2-vanilla` |
| B — +RoPE | Rotary positional embeddings (Su et al., 2022) | `out-t2-rope` |
| C — +RMSNorm+SwiGLU | RMS layer norm + gated FFN (Zhang & Sennrich, 2019; Shazeer, 2020) | `out-t2-ffn` |
| D — +QK-Norm ★ | Query-Key normalisation (Henry et al., 2020; Gemma 2, 2024) | `out-t2-qknorm` |
| E — All Modern | All three combined | `out-t2-all-modern` |

★ **Novel contribution:** QK-Norm's effect on sub-32 M narrative-generation models is not previously reported.

All experiments use identical hyperparameters (7L/6H/384D, 5K steps, lr=1e-3) for a controlled comparison.
Each model is ~31.8 M parameters, well within the 32 M constraint.

In [ ]:
import os, subprocess, json, time
os.chdir(LOCAL_DIR)

ABLATIONS = [
    ('config/train_t2_ablation_a.py', 'out-t2-vanilla',    'A. Vanilla'),
    ('config/train_t2_ablation_b.py', 'out-t2-rope',       'B. +RoPE'),
    ('config/train_t2_ablation_c.py', 'out-t2-ffn',        'C. +RMSNorm+SwiGLU'),
    ('config/train_t2_ablation_d.py', 'out-t2-qknorm',     'D. +QK-Norm (novel)'),
    ('config/train_t2_ablation_e.py', 'out-t2-all-modern', 'E. All Modern'),
]

ppl_results = {}
train_times  = {}
start_all   = time.time()
total       = len(ABLATIONS)

print(f"{'█' * 62}")
print(f"  TASK 2 — ARCHITECTURE ABLATION STUDY")
print(f"  {total} experiments  ·  ~31.8M params each  ·  5K steps each")
print(f"  Estimated total: ~{total * 6}-{total * 8} min on A100")
print(f"{'█' * 62}")

for i, (cfg, out_dir, label) in enumerate(ABLATIONS, 1):
    ckpt = os.path.join(out_dir, 'ckpt.pt')
    init = 'resume' if os.path.exists(ckpt) else 'scratch'
    elapsed_so_far = (time.time() - start_all) / 60

    print(f"\n\n{'▓' * 62}")
    print(f"  Experiment {i}/{total}  :  {label}")
    print(f"  Config          :  {cfg}")
    print(f"  init_from       :  {init}  ({'resuming' if init == 'resume' else 'scratch'})")
    print(f"  Total elapsed   :  {elapsed_so_far:.1f} min")
    print(f"{'▓' * 62}")

    # ── Train ──────────────────────────────────────────────────────────
    t0 = time.time()
    run_streaming(
        ['python', '-u', 'train.py', cfg, f'--init_from={init}'],
        label=f"[{i}/{total}] Training  {label}"
    )
    train_time = (time.time() - t0) / 60
    train_times[label] = train_time

    # ── Eval PPL ────────────────────────────────────────────────────────
    print(f"\n  ── Evaluating PPL: {label} ──")
    r = subprocess.run(
        ['python', 'eval.py', '--init_from=resume', f'--out_dir={out_dir}',
         '--input_file=data/rocstories/eval_stories.txt'],
        capture_output=True, text=True, check=False
    )
    for line in r.stdout.strip().split('\n'):
        print(f"  {line}")
    if r.returncode != 0 and r.stderr:
        print(f"  STDERR: {r.stderr[:300]}")
    for line in r.stdout.split('\n'):
        if 'ppl' in line.lower() or 'perplexity' in line.lower():
            ppl_results[label] = line.strip()

    # ── Generate samples ───────────────────────────────────────────────
    print(f"\n  ── Generating story samples: {label} ──")
    subprocess.run([
        'python', 'sample_batch.py',
        '--init_from=resume', f'--out_dir={out_dir}',
        '--start=FILE:data/rocstories/eval_prompts.txt',
        '--batch_prompts=True', '--max_new_tokens=150',
        f'--output_file={out_dir}/generated_stories.jsonl'
    ], check=False)

    print(f"\n  ✓  Experiment {i}/{total} done  —  train: {train_time:.1f} min")

# ── Final summary ──────────────────────────────────────────────────────────
total_time = (time.time() - start_all) / 60
print(f"\n\n{'═' * 62}")
print(f"  ABLATION STUDY COMPLETE  —  {total_time:.1f} min total")
print(f"{'═' * 62}")
print(f"  {'Config':28s}  {'Train (min)':>11s}  {'PPL Result'}")
print(f"  {'─' * 60}")
for lbl, train_t in train_times.items():
    ppl_line = ppl_results.get(lbl, '(not captured)')
    print(f"  {lbl:28s}  {train_t:>10.1f}  {ppl_line}")
print(f"\n  Train logs: each out-dir contains train_log.jsonl")
print(f"  Plots     : run the Analysis cell (§2.4) to generate ablation_curves.png")

██████████████████████████████████████████████████████████████
  TASK 2 — ARCHITECTURE ABLATION STUDY
  5 experiments  ·  ~30.2M params each  ·  10K steps each
  Estimated total: ~50-60 min on A100
██████████████████████████████████████████████████████████████


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  Experiment 1/5  :  A. Vanilla
  Config          :  config/train_t2_ablation_a.py
  init_from       :  scratch  (scratch)
  Total elapsed   :  0.0 min
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ▶  [1/5] Training  A. Vanilla
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Overriding config with config/train_t2_ablation_a.py:
# ============================================================
# config/train_t2_ablation_a.py
#
# TASK 2 — Ablation A: Vanilla GPT (reference point)
#
# Same architecture as Task 1 baseline but trained for
# 10,000 steps to match the other abla

<IPython.core.display.Javascript object>

step 250: train loss 4.6142, val loss 4.6805, val ppl 107.82
saving checkpoint to out-t2-vanilla/ckpt_best.pt
saving checkpoint to out-t2-vanilla/ckpt.pt
  ✓ New best val loss: 4.6805 (PPL 107.8)
  [best so far] val_loss=4.6805  PPL=107.8
iter 250: loss 4.6990, time 10662.12ms, mfu 7.07%
iter 260: loss 4.6798, time 131.24ms, mfu 7.11%
iter 270: loss 4.5998, time 127.67ms, mfu 7.17%
iter 280: loss 4.5506, time 131.07ms, mfu 7.20%
iter 290: loss 4.5880, time 128.08ms, mfu 7.24%
iter 300: loss 4.5087, time 129.24ms, mfu 7.28%
iter 310: loss 4.5123, time 129.51ms, mfu 7.31%
iter 320: loss 4.4792, time 129.24ms, mfu 7.34%
iter 330: loss 4.4366, time 132.09ms, mfu 7.34%
iter 340: loss 4.4005, time 128.84ms, mfu 7.37%
iter 350: loss 4.3691, time 129.17ms, mfu 7.39%
iter 360: loss 4.3506, time 132.71ms, mfu 7.39%
iter 370: loss 4.3204, time 131.42ms, mfu 7.40%
iter 380: loss 4.3552, time 127.31ms, mfu 7.43%
iter 390: loss 4.2621, time 133.21ms, mfu 7.42%
iter 400: loss 4.2223, time 132.39ms, m

<IPython.core.display.Javascript object>

iter 570: loss 4.0098, time 137.22ms, mfu 6.88%
iter 580: loss 3.9618, time 135.42ms, mfu 6.92%
iter 590: loss 3.9164, time 139.13ms, mfu 6.93%
iter 600: loss 3.9161, time 138.99ms, mfu 6.94%
iter 610: loss 3.9504, time 138.55ms, mfu 6.96%
iter 620: loss 3.9083, time 140.48ms, mfu 6.96%
iter 630: loss 3.8690, time 140.56ms, mfu 6.96%
iter 640: loss 3.8801, time 139.44ms, mfu 6.97%
iter 650: loss 3.8740, time 139.13ms, mfu 6.98%
iter 660: loss 3.8661, time 140.03ms, mfu 6.98%
iter 670: loss 3.8323, time 21.44ms, mfu 10.85%
iter 680: loss 3.8735, time 137.89ms, mfu 10.48%
iter 690: loss 3.8213, time 142.69ms, mfu 10.12%
iter 700: loss 3.8164, time 140.20ms, mfu 9.81%
iter 710: loss 3.7611, time 139.15ms, mfu 9.53%
iter 720: loss 3.8288, time 139.17ms, mfu 9.28%
iter 730: loss 3.7361, time 138.79ms, mfu 9.06%
iter 740: loss 3.7750, time 138.79ms, mfu 8.86%
step 750: train loss 3.6276, val loss 3.8168, val ppl 45.46
saving checkpoint to out-t2-vanilla/ckpt_best.pt
saving checkpoint to out-

...

step 4500: train loss 2.2258, val loss 3.3323, val ppl 28.00
  [best so far] val_loss=3.3218  PPL=27.7
iter 4500: loss 2.4799, time 9360.87ms, mfu 6.28%
iter 4510: loss 2.4403, time 139.50ms, mfu 6.35%
iter 4520: loss 2.4763, time 141.48ms, mfu 6.41%
iter 4530: loss 2.4751, time 142.03ms, mfu 6.46%
iter 4540: loss 2.4576, time 139.89ms, mfu 6.52%
iter 4550: loss 2.4788, time 140.69ms, mfu 6.56%
iter 4560: loss 2.5022, time 140.89ms, mfu 6.60%
iter 4570: loss 2.5214, time 141.82ms, mfu 6.63%
iter 4580: loss 2.4944, time 140.96ms, mfu 6.66%
iter 4590: loss 2.5173, time 141.57ms, mfu 6.69%
iter 4600: loss 2.4678, time 140.44ms, mfu 6.72%
iter 4610: loss 2.4568, time 142.21ms, mfu 6.74%
iter 4620: loss 2.4801, time 141.29ms, mfu 6.76%
iter 4630: loss 2.4301, time 140.22ms, mfu 6.78%
iter 4640: loss 2.5058, time 140.67ms, mfu 6.80%
iter 4650: loss 2.4508, time 140.57ms, mfu 6.82%
iter 4660: loss 2.5109, time 146.82ms, mfu 6.80%
iter 4670: loss 2.4664, time 143.00ms, mfu 6.81%
iter 4680: los

<IPython.core.display.Javascript object>

iter 4760: loss 2.4449, time 139.94ms, mfu 6.26%
iter 4770: loss 2.4748, time 142.77ms, mfu 6.32%
iter 4780: loss 2.4539, time 142.90ms, mfu 6.38%
iter 4790: loss 2.5156, time 142.80ms, mfu 6.42%
iter 4800: loss 2.4797, time 142.14ms, mfu 6.47%
iter 4810: loss 2.4477, time 143.26ms, mfu 6.51%
iter 4820: loss 2.4531, time 141.73ms, mfu 6.55%
iter 4830: loss 2.4905, time 143.21ms, mfu 6.58%
iter 4840: loss 2.4555, time 141.48ms, mfu 6.61%
iter 4850: loss 2.4808, time 144.55ms, mfu 6.63%
iter 4860: loss 2.4401, time 141.52ms, mfu 6.66%
iter 4870: loss 2.4832, time 140.27ms, mfu 6.69%
iter 4880: loss 2.4650, time 141.24ms, mfu 6.72%
iter 4890: loss 2.4760, time 143.28ms, mfu 6.73%
iter 4900: loss 2.4688, time 145.39ms, mfu 6.73%
iter 4910: loss 2.4504, time 141.04ms, mfu 6.75%
iter 4920: loss 2.4987, time 142.60ms, mfu 6.77%
iter 4930: loss 2.4750, time 141.11ms, mfu 6.78%
iter 4940: loss 2.4616, time 140.58ms, mfu 6.80%
iter 4950: loss 2.4599, time 140.05ms, mfu 6.82%
iter 4960: loss 2.43

<IPython.core.display.Javascript object>

  [keep-alive] 02:57:41
  ✓  Experiment 5/5 done  —  train: 18.9 min


══════════════════════════════════════════════════════════════
  ABLATION STUDY COMPLETE  —  96.5 min total
══════════════════════════════════════════════════════════════
  Config                        Train (min)  PPL Result
  ────────────────────────────────────────────────────────────
  A. Vanilla                          18.8  ppl             : 25.32
  B. +RoPE                            18.6  ppl             : 24.86
  C. +RMSNorm+SwiGLU                  19.2  ppl             : 24.93
  D. +QK-Norm (novel)                 19.6  ppl             : 23.43
  E. All Modern                       18.9  ppl             : 21.86

  Train logs: each out-dir contains train_log.jsonl
  Plots     : run the Analysis cell (§2.4) to generate ablation_curves.png


### 2.3 Novel Experiment — Mixed Instruction + Continuation Training

**Motivation:** Modern instruction-tuned LLMs (InstructGPT, LLaMA-chat) demonstrate that training
on both instruction prompts and raw continuations enables a single model to serve multiple
interaction modes. For small story-generation models, we hypothesise that instruction exposure
can improve narrative coherence without degrading PPL on plain continuation.

**Training mixture** (`data/mixed/train.bin`):

| Format | Proportion | Example |
|--------|-----------|----------|
| A — Plain continuation | 55% | `The boy went to a video arcade...` |
| B — Instruction-prefixed | 30% | `Write a story about: the boy went to a video arcade.\n<story>` |
| C — Structured XML | 15% | `<story><s1>...</s1>...<s5>...</s5></story>` |

The **validation set is plain ROCStories only**, ensuring PPL comparisons are directly comparable
to Task 1 and the ablation baselines.

**Reference:** Ouyang, L. et al. (2022). Training language models to follow instructions with
human feedback. arXiv:2203.02155.

In [ ]:
import os
os.chdir(LOCAL_DIR)

T3_CONFIG  = 'config/train_t3_best.py'
T3_OUT_DIR = 'out-t3-best'

ckpt = os.path.join(T3_OUT_DIR, 'ckpt.pt')
init_t3 = 'resume' if os.path.exists(ckpt) else 'scratch'

print(f"  Model    : All-Modern 30M  (RoPE + RMSNorm + SwiGLU + QK-Norm)")
print(f"  Dataset  : data/rocstories/  (val = eval_stories.txt — professor's eval set)")
print(f"  Config   : {T3_CONFIG}")
print(f"  Out dir  : {T3_OUT_DIR}")
print(f"  Init     : {init_t3}  ({'resuming from checkpoint' if init_t3 == 'resume' else 'training from scratch'})")
# print(f"  Steps    : 15,000  (longer than ablations for best PPL)")
print(f"  Steps    : 8,000  (longer than ablations; val = eval_stories.txt for exact PPL match)")
print(f"  ETA      : ~12–15 min on A100\n")
print(f"  NOTE: this model is also the Task 3 HuggingFace submission.")

rc = run_streaming(
    ['python', '-u', 'train.py', T3_CONFIG, f'--init_from={init_t3}'],
    label="Task 3 Best Submission  ·  All-Modern 30M  —  ROCStories (pure)"
)

if rc == 0:
    print(f"\n✓  Training complete.  Checkpoint saved to: {T3_OUT_DIR}")
else:
    print(f"\n⚠  Process exited with code {rc}.")
    print("   Re-run this cell — it will resume from the last checkpoint automatically.")

  Model    : All-Modern 30M  (RoPE + RMSNorm + SwiGLU + QK-Norm)
  Dataset  : data/mixed/  (100% plain · 0% instruction · 0% structured)
  Config   : config/train_t3_best.py
  Out dir  : out-t3-best
  Init     : scratch  (training from scratch)
  Steps    : 20,000  (longer than ablations for best PPL)
  ETA      : ~12–15 min on A100

  NOTE: this model is also the Task 3 HuggingFace submission.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ▶  Task 3 / Instruction Experiment  ·  All-Modern 30M + Mixed Data
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Overriding config with config/train_t3_best.py:
# ============================================================
# config/train_t3_best.py
#
# TASK 3 — Best Checkpoint Submission (≤ 32M params)
#
# Architecture: All-Modern 30M (RoPE + RMSNorm + SwiGLU + QK-Norm)
#   → Best configuration identified in the Task 2 ablation study
#
# Data: Mixed instruction dataset (data/mixed/)
#   55% plain continuation (w

<IPython.core.display.Javascript object>

iter 180: loss 3.3398, time 140.84ms, mfu 6.90%
iter 190: loss 3.3293, time 141.12ms, mfu 6.91%
iter 200: loss 3.1902, time 141.38ms, mfu 6.91%
iter 210: loss 3.2200, time 147.89ms, mfu 6.88%
iter 220: loss 3.1994, time 141.65ms, mfu 6.88%
iter 230: loss 3.1882, time 144.33ms, mfu 6.88%
iter 240: loss 3.1600, time 142.28ms, mfu 6.88%
step 250: train loss 3.0051, val loss 4.9319, val ppl 138.64
saving checkpoint to out-t3-best/ckpt_best.pt
saving checkpoint to out-t3-best/ckpt.pt
  ✓ New best val loss: 4.9319 (PPL 138.6)
  [best so far] val_loss=4.9319  PPL=138.6
iter 250: loss 3.1344, time 11199.64ms, mfu 6.20%
iter 260: loss 3.1352, time 138.61ms, mfu 6.29%
iter 270: loss 3.0545, time 141.04ms, mfu 6.35%
iter 280: loss 2.9241, time 139.96ms, mfu 6.42%
iter 290: loss 2.9646, time 141.37ms, mfu 6.47%
iter 300: loss 3.0223, time 141.85ms, mfu 6.51%
iter 310: loss 2.8906, time 140.06ms, mfu 6.56%
iter 320: loss 2.8781, time 141.81ms, mfu 6.60%
iter 330: loss 2.9247, time 142.03ms, mfu 6.6

<IPython.core.display.Javascript object>

iter 490: loss 2.6101, time 141.59ms, mfu 6.88%
step 500: train loss 2.5508, val loss 4.4876, val ppl 88.90
saving checkpoint to out-t3-best/ckpt_best.pt
saving checkpoint to out-t3-best/ckpt.pt
  ✓ New best val loss: 4.4876 (PPL 88.9)
  [best so far] val_loss=4.4876  PPL=88.9
iter 500: loss 2.5362, time 11560.24ms, mfu 6.20%
iter 510: loss 2.6940, time 140.15ms, mfu 6.28%
iter 520: loss 2.4525, time 141.46ms, mfu 6.35%
iter 530: loss 2.5761, time 141.87ms, mfu 6.40%
iter 540: loss 2.5138, time 141.06ms, mfu 6.46%
iter 550: loss 2.7116, time 141.56ms, mfu 6.50%
iter 560: loss 2.4855, time 142.77ms, mfu 6.54%
iter 570: loss 2.6013, time 25.46ms, mfu 9.74%
iter 580: loss 2.6015, time 138.44ms, mfu 9.47%
iter 590: loss 2.5940, time 142.55ms, mfu 9.21%
iter 600: loss 2.4663, time 142.32ms, mfu 8.98%
iter 610: loss 2.5396, time 140.65ms, mfu 8.78%
iter 620: loss 2.5007, time 140.63ms, mfu 8.60%
iter 630: loss 2.4867, time 141.60ms, mfu 8.43%
iter 640: loss 2.5014, time 140.26ms, mfu 8.29%
i

...

iter 19380: loss 1.7852, time 140.51ms, mfu 6.73%
iter 19390: loss 1.7528, time 141.36ms, mfu 6.75%
iter 19400: loss 1.7202, time 139.58ms, mfu 6.78%
iter 19410: loss 1.6946, time 141.56ms, mfu 6.79%
iter 19420: loss 1.6729, time 139.93ms, mfu 6.81%
iter 19430: loss 1.7468, time 141.97ms, mfu 6.82%
iter 19440: loss 1.7518, time 140.88ms, mfu 6.84%
iter 19450: loss 1.6714, time 142.82ms, mfu 6.84%
iter 19460: loss 1.7645, time 141.26ms, mfu 6.85%
iter 19470: loss 1.7504, time 140.80ms, mfu 6.86%
iter 19480: loss 1.7542, time 142.18ms, mfu 6.86%
iter 19490: loss 1.7409, time 141.04ms, mfu 6.87%
step 19500: train loss 1.6139, val loss 3.4031, val ppl 30.06
saving checkpoint to out-t3-best/ckpt_best.pt
saving checkpoint to out-t3-best/ckpt.pt
  ✓ New best val loss: 3.4031 (PPL 30.1)
  [best so far] val_loss=3.4031  PPL=30.1
iter 19500: loss 1.8225, time 11839.18ms, mfu 6.19%
iter 19510: loss 1.7397, time 139.70ms, mfu 6.28%
iter 19520: loss 1.6730, time 140.30ms, mfu 6.35%
iter 19530: loss

<IPython.core.display.Javascript object>

iter 19640: loss 1.8521, time 141.32ms, mfu 6.77%
iter 19650: loss 1.7255, time 145.51ms, mfu 6.77%
iter 19660: loss 1.6819, time 141.18ms, mfu 6.79%
iter 19670: loss 1.7550, time 140.58ms, mfu 6.81%
iter 19680: loss 1.6816, time 141.46ms, mfu 6.82%
iter 19690: loss 1.8058, time 141.96ms, mfu 6.83%
iter 19700: loss 1.6795, time 150.09ms, mfu 6.80%
iter 19710: loss 1.7660, time 146.87ms, mfu 6.79%
iter 19720: loss 1.7698, time 143.26ms, mfu 6.79%
iter 19730: loss 1.6321, time 141.93ms, mfu 6.80%
iter 19740: loss 1.6980, time 142.17ms, mfu 6.81%
step 19750: train loss 1.6111, val loss 3.4038, val ppl 30.08
  [best so far] val_loss=3.4031  PPL=30.1
iter 19750: loss 1.7547, time 9923.68ms, mfu 6.14%
iter 19760: loss 1.7795, time 147.85ms, mfu 6.19%
iter 19770: loss 1.6892, time 147.68ms, mfu 6.24%
iter 19780: loss 1.6670, time 148.26ms, mfu 6.27%
iter 19790: loss 1.6566, time 143.59ms, mfu 6.33%
iter 19800: loss 1.7421, time 141.02ms, mfu 6.39%
iter 19810: loss 1.7178, time 140.36ms, mfu 6

<IPython.core.display.Javascript object>

iter 19910: loss 1.6939, time 141.80ms, mfu 6.75%
iter 19920: loss 1.6192, time 140.52ms, mfu 6.77%
iter 19930: loss 1.7911, time 140.43ms, mfu 6.79%
iter 19940: loss 1.8001, time 142.98ms, mfu 6.80%
iter 19950: loss 1.6189, time 140.62ms, mfu 6.82%
iter 19960: loss 1.7497, time 141.57ms, mfu 6.83%
iter 19970: loss 1.7087, time 141.50ms, mfu 6.84%
iter 19980: loss 1.8021, time 141.10ms, mfu 6.85%
iter 19990: loss 1.7176, time 141.54ms, mfu 6.86%
step 20000: train loss 1.6035, val loss 3.4088, val ppl 30.23
  [best so far] val_loss=3.4031  PPL=30.1
iter 20000: loss 1.6807, time 9613.77ms, mfu 6.18%
Training complete — saving final checkpoint
saving checkpoint to out-t3-best/ckpt_final.pt
saving checkpoint to out-t3-best/ckpt.pt

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✓  finished in 73m 32s
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✓  Training complete.  Checkpoint saved to: out-t3-best


### 2.4 Analysis and Visualisation

The following cell generates:
1. **Learning-curve overlay** — val loss across all 5 architecture ablations
2. **Bar chart** — final val loss per experiment
3. **Story quality comparison** — lexical diversity and avg length per model

All outputs are saved as `.png` files for the report.

In [6]:
import os, json, subprocess
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
os.chdir(LOCAL_DIR)

# ── Discover completed experiments ───────────────────────────────────────────
ALL_EXPS = [
    ('out-t1-baseline',   'Task 1 Baseline'),
    ('out-t2-vanilla',    'A. Vanilla'),
    ('out-t2-rope',       'B. +RoPE'),
    ('out-t2-ffn',        'C. +RMSNorm+SwiGLU'),
    ('out-t2-qknorm',     'D. +QK-Norm'),
    ('out-t2-all-modern', 'E. All Modern'),
    ('out-t3-best',       'F. T3 Best (pure ROCStories)'),
]

available = [
    (d, l, os.path.join(d, 'train_log.jsonl'))
    for d, l in ALL_EXPS
    if os.path.exists(os.path.join(d, 'train_log.jsonl'))
]

print(f"Found {len(available)}/{len(ALL_EXPS)} completed experiments:")
for d, l, p in available:
    print(f"  \u2713 {l:28s} — {p}")
missing = [(d, l) for d, l in ALL_EXPS
           if not os.path.exists(os.path.join(d, 'train_log.jsonl'))]
for d, l in missing:
    print(f"  \u2717 {l:28s} — not yet run")

if not available:
    print("\nNo completed experiments yet. Run the training cells first.")
else:
    COLORS = ['#555555','#E53935','#1E88E5','#43A047','#FB8C00','#8E24AA','#00ACC1']

    # ── 1. Learning-curve overlay ─────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    for (d, label, log_path), color in zip(available, COLORS):
        steps, val_losses = [], []
        with open(log_path) as f:
            for line in f:
                entry = json.loads(line)
                if entry.get('val_loss') is not None:
                    steps.append(entry['step'])
                    val_losses.append(entry['val_loss'])
        if steps:
            ax.plot(steps, val_losses, label=label, color=color, linewidth=1.8)
    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Validation Loss', fontsize=12)
    ax.set_title('Task 2: Validation Loss — Architecture Ablation', fontsize=14, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('ablation_curves.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("\n\u2713 Saved: ablation_curves.png")

    # ── 2. Bar chart of final val loss ─────────────────────────────────────
    bar_labels, bar_vals = [], []
    for d, label, log_path in available:
        with open(log_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        for line in reversed(lines):
            entry = json.loads(line)
            if entry.get('val_loss') is not None:
                bar_labels.append(label)
                bar_vals.append(entry['val_loss'])
                break

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(bar_labels, bar_vals, color=COLORS[:len(bar_vals)],
                  edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, bar_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylabel('Final Validation Loss', fontsize=12)
    ax.set_title('Task 2: Final Validation Loss by Configuration', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=20, ha='right', fontsize=9)
    plt.tight_layout()
    plt.savefig('ablation_bar_chart.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("\u2713 Saved: ablation_bar_chart.png")

    # ── 3. Summary table ──────────────────────────────────────────────────
    print(f"\n  {'Config':30s} {'Final Val Loss':>15s}")
    print(f"  {'─'*47}")
    for lbl, val in sorted(zip(bar_labels, bar_vals), key=lambda x: x[1]):
        print(f"  {lbl:30s} {val:15.4f}")

    # ── 4. Story quality (lexical diversity) ─────────────────────────────
    print(f"\n  {'Config':30s} {'Stories':>8s} {'Avg Len':>8s} {'Type/Token':>12s}")
    print(f"  {'─'*60}")
    for d, label, _ in available:
        stories_path = os.path.join(d, 'generated_stories.jsonl')
        if not os.path.exists(stories_path):
            continue
        texts = []
        with open(stories_path) as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    texts.append(obj.get('generated_text', obj.get('text', '')))
                except json.JSONDecodeError:
                    pass
        if not texts:
            continue
        all_words = ' '.join(texts).lower().split()
        ttr = len(set(all_words)) / max(len(all_words), 1)
        avg_len = sum(len(t.split()) for t in texts) / len(texts)
        print(f"  {label:30s} {len(texts):8d} {avg_len:8.1f} {ttr:12.4f}")

print(f"\n  Download ablation_curves.png and ablation_bar_chart.png for your report.")

Found 7/7 completed experiments:
  ✓ Task 1 Baseline              — out-t1-baseline/train_log.jsonl
  ✓ A. Vanilla                   — out-t2-vanilla/train_log.jsonl
  ✓ B. +RoPE                     — out-t2-rope/train_log.jsonl
  ✓ C. +RMSNorm+SwiGLU           — out-t2-ffn/train_log.jsonl
  ✓ D. +QK-Norm                  — out-t2-qknorm/train_log.jsonl
  ✓ E. All Modern                — out-t2-all-modern/train_log.jsonl
  ✓ F. +Mixed Instr.             — out-t3-best/train_log.jsonl

✓ Saved: ablation_curves.png
✓ Saved: ablation_bar_chart.png

  Config                          Final Val Loss
  ───────────────────────────────────────────────
  D. +QK-Norm                             3.3304
  E. All Modern                           3.3305
  B. +RoPE                                3.3352
  A. Vanilla                              3.3529
  Task 1 Baseline                         3.3604
  C. +RMSNorm+SwiGLU                      3.3759
  F. +Mixed Instr.                        3.4088

  Conf

---
## §3 · Task 3 — Best Checkpoint Submission

The best submission model is the **all-modern 31.8 M trained on pure ROCStories** (`out-t3-best/ckpt_best.pt`).
Val loss was monitored against `eval_stories.txt` — the professor's exact evaluation set — ensuring
`ckpt_best.pt` is the checkpoint with the lowest PPL on that set. No new training is needed here.

### 3.1 Final Evaluation

Comprehensive evaluation on the full public test set.

In [8]:
import os, subprocess
os.chdir(LOCAL_DIR)

print("=" * 55)
print("TASK 3 — Final PPL on test set")
print("=" * 55)
subprocess.run([
    'python', 'eval.py',
    '--init_from=resume',
    '--out_dir=out-t3-best',
    '--input_file=data/rocstories/eval_stories.txt'
], check=False)

print()
print("=" * 55)
print("TASK 3 — Story generation (plain prompts)")
print("=" * 55)
subprocess.run([
    'python', 'sample_batch.py',
    '--init_from=resume', '--out_dir=out-t3-best',
    '--start=FILE:data/rocstories/eval_prompts.txt',
    '--batch_prompts=True', '--max_new_tokens=200',
    '--output_file=out-t3-best/generated_stories.jsonl'
], check=False)

print()
print("=" * 55)
print("TASK 3 — Story generation (instruction prompts)")
print("=" * 55)
# Also test instruction-mode generation to verify dual capability
instr_prompts = [
    'Write a story about: a girl who lost her dog.\n',
    'Tell me a 5-sentence story involving a birthday surprise.\n',
    'Generate a brief narrative about learning to ride a bike.\n',
]
import json as _json
with open('/tmp/instr_prompts.txt', 'w') as f:
    f.write('\n---\n'.join(instr_prompts))
subprocess.run([
    'python', 'sample_batch.py',
    '--init_from=resume', '--out_dir=out-t3-best',
    '--start=FILE:/tmp/instr_prompts.txt',
    '--batch_prompts=True', '--max_new_tokens=200',
    '--output_file=out-t3-best/generated_instruction_stories.jsonl'
], check=False)

TASK 3 — Final PPL on test set

TASK 3 — Story generation (plain prompts)

TASK 3 — Story generation (instruction prompts)


CompletedProcess(args=['python', 'sample_batch.py', '--init_from=resume', '--out_dir=out-t3-best', '--start=FILE:/tmp/instr_prompts.txt', '--batch_prompts=True', '--max_new_tokens=200', '--output_file=out-t3-best/generated_instruction_stories.jsonl'], returncode=1)

In [9]:
import json

with open('out-t3-best/generated_stories.jsonl') as f:
    for i, line in enumerate(f):
        print(json.loads(line))
        if i == 2:
            break

{'prompt': 'Emily forgot her umbrella before leaving for work.', 'generated_text': 'Emily forgot her umbrella before leaving for work. She wanted to stay outside and play in the park, but she remembered that it was important to listen to her mommy and not go outside without permission.', 'params': {'max_new_tokens': 200, 'temperature': 0.75, 'top_k': 50, 'top_p': 0.9, 'repetition_penalty': 1.2}}
{'prompt': 'Tom decided to cook dinner for his friends.', 'generated_text': 'Tom decided to cook dinner for his friends. They all sat down at the table and enjoyed their meal together. The sun was shining and the sky was blue, but Tom had a problem with his fork. He couldn\'t eat it because he was too small to fit in. \n\nHis friend Sam came over and saw that Tom was sad. "What\'s wrong, Tom?" asked Sam. "I want my fork back," replied Tom. Sam felt bad for making Tom cry. "Don\'t worry, I\'ll help you find some more fork," said Sam. So they went off to look for another spoon.', 'params': {'max_

### 3.2 Package and Upload to HuggingFace

Files uploaded per assignment specification (`hf_load.py`):
- `ckpt.pt` — final checkpoint
- `model.py` — modified architecture (with RoPE, RMSNorm, SwiGLU, QK-Norm)
- `sample_params.json` — generation hyperparameters used for sampling
- `eval.py` + `configurator.py` — for evaluator to load and run the model

**Submission repo:** `<HF_USERNAME>/nanoGPT_hw`

In [ ]:
# ── §3.4 HuggingFace Submission ───────────────────────────────────────────
import os, shutil, json, math, torch, subprocess
from google.colab import userdata
os.chdir(LOCAL_DIR)

HF_USERNAME = "YOUR_HF_USERNAME"   # ← REPLACE
HF_TOKEN    = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

T3_DIR     = 'out-t3-best'
SUBMIT_DIR = 'submission_hf'
os.makedirs(SUBMIT_DIR, exist_ok=True)

# Use best checkpoint (not final — avoids the overfitting trap)
best_ckpt  = os.path.join(T3_DIR, 'ckpt_best.pt')
final_ckpt = os.path.join(T3_DIR, 'ckpt.pt')
src_ckpt   = best_ckpt if os.path.exists(best_ckpt) else final_ckpt
print(f"Using: {src_ckpt}")

# Verify before submitting
ck       = torch.load(src_ckpt, map_location='cpu', weights_only=False)
best_ppl = math.exp(ck['best_val_loss'])
from model import GPT, GPTConfig
m = GPT(GPTConfig(**ck['model_args']))
n = sum(p.numel() for p in m.parameters())
print(f"  Best val PPL: {best_ppl:.1f}")
print(f"  Parameters:   {n/1e6:.2f}M {'✓' if n <= 32e6 else '✗ OVER 32M LIMIT'}")
assert n <= 32_000_000, "Model exceeds 32M param limit!"
assert best_ppl < 40, f"PPL {best_ppl:.1f} seems too high — check dataset and checkpoint"

# Write generation params
params = {"temperature": 0.75, "top_k": 50, "top_p": 0.9, "repetition_penalty": 1.2}
with open('sample_params.json', 'w') as f:
    json.dump(params, f, indent=2)

# Copy all required files
for src, dst in [
    (src_ckpt,            f'{SUBMIT_DIR}/ckpt.pt'),   # must be named ckpt.pt
    ('model.py',          f'{SUBMIT_DIR}/model.py'),
    ('sample_params.json',f'{SUBMIT_DIR}/sample_params.json'),
    ('eval.py',           f'{SUBMIT_DIR}/eval.py'),
    ('configurator.py',   f'{SUBMIT_DIR}/configurator.py'),
]:
    if os.path.exists(src):
        shutil.copy(src, dst)

print("\nSubmission folder contents:")
for fname in sorted(os.listdir(SUBMIT_DIR)):
    mb = os.path.getsize(f'{SUBMIT_DIR}/{fname}') / 1e6
    print(f"  {fname:<30s} {mb:.1f} MB")

REPO_ID = f"{HF_USERNAME}/nanoGPT_hw"
result = subprocess.run([
    'python', 'hf_load.py', 'upload',
    '--local-dir', SUBMIT_DIR,
    '--repo-id',   REPO_ID,
    '--token',     HF_TOKEN,
    '--commit-message', f'Best nanoGPT checkpoint — {n/1e6:.1f}M params, best PPL {best_ppl:.1f}'
], check=False)

print(f"\n✓ Uploaded to: https://huggingface.co/{REPO_ID}")
print(f"  Canvas submission: {REPO_ID}")

---
## §4 · Task 4 — Arena Competition (Optional, 124 M)

> **Note:** This section is entirely optional and only relevant for the **arena competition**.
> The 124 M model **must NOT be submitted** to HuggingFace for Task 3 grading — it exceeds the 32 M constraint.

The arena model uses the same LLaMA-style architecture scaled to 124 M (12L/12H/768D).
Training is two-stage: Stage 1 pretrains on the combined ROCStories + TinyStories dataset
(~54 M tokens); Stage 2 fine-tunes on instruction-format stories for arena judging.

### Generation strategy for human judging:
- Temperature: 0.85 (slightly higher for more creative, varied stories)
- Top-K: 50, Top-P: 0.9
- Repetition penalty: 1.2 (prevents looping, critical for coherence)

In [ ]:
import os, numpy as np
os.chdir(LOCAL_DIR)

# Prepare combined dataset (ROCStories + TinyStories)
if not os.path.exists('data/combined/train.bin'):
    print("Building combined dataset...")
    !python data/combined/prepare.py
else:
    arr = np.fromfile('data/combined/train.bin', dtype=np.uint16)
    print(f"Combined dataset already exists: {len(arr)/1e6:.0f}M tokens")

In [ ]:
import os
os.chdir(LOCAL_DIR)

# ── Stage 1: Pretrain (124M on combined corpus) ───────────────────────────
T4_S1_CONFIG  = 'config/train_t4_arena.py'
T4_S1_OUT_DIR = 'out-t4-pretrain'

ckpt_s1 = os.path.join(T4_S1_OUT_DIR, 'ckpt.pt')
init_t4 = 'resume' if os.path.exists(ckpt_s1) else 'scratch'

print(f"  ⚠  DO NOT upload this model for Task 3 — it exceeds 32M params!")
print(f"  This model is for the optional arena competition only.\n")
print(f"  ── Stage 1: Pretrain ───────────────────────────────────────────")
print(f"  Model    : 124M  (12L/12H/768D, all-modern LLaMA-style)")
print(f"  Dataset  : data/combined/  (ROCStories + TinyStories, ~54M tokens)")
print(f"  Config   : {T4_S1_CONFIG}")
print(f"  Out dir  : {T4_S1_OUT_DIR}")
print(f"  Init     : {init_t4}  ({'resuming from checkpoint' if init_t4 == 'resume' else 'training from scratch'})")
print(f"  Steps    : 20,000")
print(f"  ETA      : ~60–90 min on A100\n")

rc = run_streaming(
    ['python', '-u', 'train.py', T4_S1_CONFIG, f'--init_from={init_t4}'],
    label="Task 4 Stage 1  ·  124M Pretrain  —  ROCStories + TinyStories"
)

if rc == 0:
    print(f"\n✓  Stage 1 pretrain complete.  Checkpoint saved to: {T4_S1_OUT_DIR}")
else:
    print(f"\n⚠  Process exited with code {rc}.")
    print("   Re-run this cell — it will resume from the last checkpoint automatically.")
    raise RuntimeError("Stage 1 failed — fix before proceeding to Stage 2")

# ── Copy best pretrain checkpoint for Stage 2 to resume from ─────────────
import shutil
T4_S2_OUT_DIR = 'out-t4-arena'
os.makedirs(T4_S2_OUT_DIR, exist_ok=True)

best_s1 = os.path.join(T4_S1_OUT_DIR, 'ckpt_best.pt')
resume_s2 = os.path.join(T4_S2_OUT_DIR, 'ckpt.pt')

if os.path.exists(best_s1):
    shutil.copy(best_s1, resume_s2)
    print(f"\n✓  Copied {T4_S1_OUT_DIR}/ckpt_best.pt → {T4_S2_OUT_DIR}/ckpt.pt")
else:
    # Fallback to ckpt.pt
    shutil.copy(os.path.join(T4_S1_OUT_DIR, 'ckpt.pt'), resume_s2)
    print(f"\n⚠  ckpt_best.pt not found — copied ckpt.pt instead")

# ── Stage 2: Instruction fine-tune ───────────────────────────────────────
T4_S2_CONFIG = 'config/train_t4_finetune.py'

print(f"\n  ── Stage 2: Instruction Fine-tune ───────────────────────────────")
print(f"  Dataset  : data/rocstories_instruction/")
print(f"  Config   : {T4_S2_CONFIG}")
print(f"  Out dir  : {T4_S2_OUT_DIR}")
print(f"  Init     : resume (from Stage 1 best checkpoint)")
print(f"  Steps    : 2,000")
print(f"  ETA      : ~8 min on A100\n")

rc2 = run_streaming(
    ['python', '-u', 'train.py', T4_S2_CONFIG],
    label="Task 4 Stage 2  ·  124M Instruction Fine-tune"
)

if rc2 == 0:
    print(f"\n✓  Stage 2 fine-tune complete.  Final model saved to: {T4_S2_OUT_DIR}")
else:
    print(f"\n⚠  Stage 2 exited with code {rc2}.")
    print("   Re-run from this cell — it will resume automatically.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §4.3 — Task 4: Evaluation, Inference & Charts
# ═══════════════════════════════════════════════════════════════════════════
import json, math, os, torch
import matplotlib.pyplot as plt
import numpy as np
import tiktoken
os.chdir(LOCAL_DIR)

# ── 4.3.1 Load T4 checkpoint ──────────────────────────────────────────────
T4_DIR = 'out-t4-arena'
ckpt_path = os.path.join(T4_DIR, 'ckpt_best.pt')
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(T4_DIR, 'ckpt.pt')

ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
from model import GPT, GPTConfig
cfg = GPTConfig(**ckpt['model_args'])
t4_model = GPT(cfg)
t4_model.load_state_dict(ckpt['model'])
t4_model.eval()

n_params = sum(p.numel() for p in t4_model.parameters())
print(f"T4 Arena model loaded")
print(f"  Architecture: {cfg.n_layer}L / {cfg.n_head}H / {cfg.n_embd}D")
print(f"  Parameters:   {n_params/1e6:.1f}M")
print(f"  Best val loss: {ckpt['best_val_loss']:.4f}  (PPL {math.exp(ckpt['best_val_loss']):.1f})")
print(f"  Trained for:  {ckpt['iter_num']} steps")

# ── 4.3.2 PPL on eval stories ─────────────────────────────────────────────
enc = tiktoken.get_encoding('gpt2')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
t4_model = t4_model.to(device)

eval_prompts = [
    "Emily forgot her umbrella before leaving for work.",
    "Tom decided to cook dinner for his friends.",
    "Lily wanted to start jogging every morning.",
    "Mark borrowed his sister's bike for the afternoon.",
    "Anna planted tomato seeds in her backyard.",
]

def eval_ppl_on_text(model, texts, enc, device):
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for text in texts:
            ids = enc.encode_ordinary(text)
            if len(ids) < 2: continue
            x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
            y = torch.tensor([ids[1:]],  dtype=torch.long, device=device)
            _, loss = model(x, y)
            total_loss   += loss.item() * (len(ids) - 1)
            total_tokens += (len(ids) - 1)
    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    return avg_loss, math.exp(avg_loss)

eval_stories_path = 'data/rocstories/eval_stories.txt'
if os.path.exists(eval_stories_path):
    with open(eval_stories_path) as f:
        raw = f.read().strip()
    eval_texts = [s.strip() for s in raw.split('\n\n') if s.strip()]
else:
    eval_texts = eval_prompts
avg_loss_t4, ppl_t4 = eval_ppl_on_text(t4_model, eval_texts, enc, device)
print(f"\nT4 eval PPL on {len(eval_texts)} stories: {ppl_t4:.2f}")

# ── 4.3.3 Story generation ────────────────────────────────────────────────
T4_PARAMS = dict(max_new_tokens=200, temperature=0.75, top_k=50,
                 top_p=0.9, repetition_penalty=1.3)

print("\n── T4 Arena Generated Stories ──────────────────────────────────────")
t4_stories = []
for prompt in eval_prompts:
    ids = enc.encode_ordinary(prompt)
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = t4_model.generate(idx, **T4_PARAMS)
    text = enc.decode(out[0].tolist())
    t4_stories.append({'prompt': prompt, 'generated': text})
    print(f"\nPrompt: {prompt}")
    print(f"Story:  {text[:300]}")

# ── 4.3.4 Learning curves ─────────────────────────────────────────────────
log_path = os.path.join(T4_DIR, 'train_log.jsonl')
if os.path.exists(log_path):
    t4_log = [json.loads(l) for l in open(log_path)]
    t4_val = [(e['step'], e['val_loss']) for e in t4_log if 'val_loss' in e]
    t4_train_steps = [e['step'] for e in t4_log if 'train_loss' in e and 'val_loss' not in e]
    t4_train_loss  = [e['train_loss'] for e in t4_log if 'train_loss' in e and 'val_loss' not in e]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Task 4 — Arena Model (124M, Stage 1 pretrain + Stage 2 fine-tune)', fontsize=13, fontweight='bold', y=1.02)

    ax = axes[0]
    if t4_train_steps:
        ax.plot(t4_train_steps, t4_train_loss, alpha=0.4, color='#B5D4F4', label='Train loss', linewidth=1)
    if t4_val:
        ax.plot([v[0] for v in t4_val], [v[1] for v in t4_val],
                color='#378ADD', label='Val loss', linewidth=2)
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title('Training curves')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax2 = axes[1]
    if t4_val:
        val_ppls  = [math.exp(v[1]) for v in t4_val]
        val_steps = [v[0] for v in t4_val]
        ax2.plot(val_steps, val_ppls, color='#1D9E75', linewidth=2)
        best_step = val_steps[val_ppls.index(min(val_ppls))]
        best_ppl  = min(val_ppls)
        ax2.axvline(best_step, color='#E24B4A', linestyle='--', alpha=0.6, label=f'Best PPL={best_ppl:.1f}')
        ax2.axhline(25, color='#BA7517', linestyle=':', alpha=0.8, label='Pass line (25)')
    ax2.set_xlabel('Step'); ax2.set_ylabel('Perplexity')
    ax2.set_title('Val PPL over training')
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    ax3 = axes[2]
    model_labels, model_ppls, model_colors = [], [], []
    for out_dir, label, color in [
        ('out-t1-baseline', 'T1 Baseline\n(30M)', '#888780'),
        ('out-t3-best',     'T3 Best\n(30M)',     '#378ADD'),
        (T4_DIR,            'T4 Arena\n(152M)',   '#1D9E75'),
    ]:
        lp = os.path.join(out_dir, 'train_log.jsonl')
        if os.path.exists(lp):
            entries = [json.loads(l) for l in open(lp) if 'val_loss' in l]
            if entries:
                best_vl = min(entries, key=lambda e: e['val_loss'])['val_loss']
                model_labels.append(label)
                model_ppls.append(math.exp(best_vl))
                model_colors.append(color)
    if model_ppls:
        bars = ax3.bar(model_labels, model_ppls, color=model_colors, width=0.5)
        ax3.axhline(25, color='#E24B4A', linestyle='--', alpha=0.8, label='PPL=25 pass line')
        for bar, ppl in zip(bars, model_ppls):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{ppl:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax3.set_ylabel('Best Val PPL'); ax3.set_title('T1 vs T3 vs T4 comparison')
    ax3.legend(fontsize=9); ax3.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('t4_evaluation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: t4_evaluation.png")

In [ ]:
# Run this cell in PARALLEL with any long training cell to prevent Colab idle disconnect
import time, threading
from IPython.display import display, Javascript

def keep_alive():
    while True:
        time.sleep(55)
        try:
            display(Javascript('console.log("keep-alive")'))
        except Exception:
            pass
        print(f"  [keep-alive] {time.strftime('%H:%M:%S')}", end="\r")

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print("\u2713 Keep-alive thread started (pings every 55 s)")

✓ Keep-alive thread started (pings every 55 s)


---
## §5 · Final Summary — All Tasks

Aggregates results from Tasks 1–4. Run after all training is complete.

| Task | Model | Params | Expected Best PPL | Notes |
|------|-------|--------|-------------------|-------|
| T1 | Baseline (5K steps) | ~31.8M | ~22–25 | Pure ROCStories, 7L |
| T2-D | +QK-Norm ★ | ~31.8M | **~21–24** | Best individual modification |
| T3 | All Modern (8K steps, rocstories) | ~31.8M | **~19–22** | Val = eval_stories.txt |
| T4 | 124M (20K pretrain + 2K fine-tune) | 124M | N/A (arena) | Human-judged, no size limit |

*PPL < 25 threshold applies to T3 HuggingFace submission only.*

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §5 — FINAL SUMMARY: Task 1 → Task 4
# ═══════════════════════════════════════════════════════════════════════════
import json, math, os, torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
os.chdir(LOCAL_DIR)

# ── Collect all results ────────────────────────────────────────────────────
ALL_RUNS = [
    ('out-t1-baseline',   'T1: Baseline',         '#888780', False, False, False, False),
    ('out-t2-vanilla',    'T2-A: Vanilla',         '#888780', False, False, False, False),
    ('out-t2-rope',       'T2-B: +RoPE',           '#378ADD', True,  False, False, False),
    ('out-t2-ffn',        'T2-C: +RMSNorm+SwiGLU','#1D9E75', False, True,  True,  False),
    ('out-t2-qknorm',     'T2-D: +QK-Norm ★',     '#BA7517', False, False, False, True ),
    ('out-t2-all-modern', 'T2-E: All Modern',      '#7F77DD', True,  True,  True,  True ),
    ('out-t3-best',       'T3: Best Submission',   '#D85A30', True,  True,  True,  True ),
    ('out-t4-arena',      'T4: Arena (124M)',        '#0F6E56', True,  True,  True,  True ),
]

results = []
for out_dir, label, color, rope, rmsnorm, swiglu, qknorm in ALL_RUNS:
    log = os.path.join(out_dir, 'train_log.jsonl')
    if not os.path.exists(log):
        results.append({'label': label, 'color': color, 'best_ppl': None,
                        'best_step': None, 'rope': rope, 'rmsnorm': rmsnorm,
                        'swiglu': swiglu, 'qknorm': qknorm})
        continue
    entries = [json.loads(l) for l in open(log) if 'val_loss' in l]
    if not entries:
        continue
    best = min(entries, key=lambda e: e['val_loss'])
    ckpt_path = os.path.join(out_dir, 'ckpt_best.pt')
    if not os.path.exists(ckpt_path):
        ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    n_params = None
    if os.path.exists(ckpt_path):
        try:
            ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            from model import GPT, GPTConfig
            m = GPT(GPTConfig(**ck['model_args']))
            n_params = sum(p.numel() for p in m.parameters()) / 1e6
        except: pass
    results.append({
        'label': label, 'color': color,
        'best_ppl': math.exp(best['val_loss']),
        'best_step': best['step'],
        'best_val': best['val_loss'],
        'n_params': n_params,
        'rope': rope, 'rmsnorm': rmsnorm, 'swiglu': swiglu, 'qknorm': qknorm,
    })

# ── Print summary table ────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║              FINAL RESULTS SUMMARY — All Tasks                      ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  {'Model':<30} {'Params':>7} {'Best PPL':>9} {'Best Step':>10} {'Flags':<15} ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
for r in results:
    if r['best_ppl'] is None:
        print(f"║  {r['label']:<30} {'—':>7} {'not run':>9} {'—':>10} {'—':<15} ║")
        continue
    params = f"{r['n_params']:.0f}M" if r.get('n_params') else "—"
    flags = '+'.join([f for f in [
        'R' if r['rope'] else '',
        'N' if r['rmsnorm'] else '',
        'S' if r['swiglu'] else '',
        'Q' if r['qknorm'] else '',
    ] if f])
    flags = flags if flags else 'vanilla'
    pass_marker = ' ✓' if r['best_ppl'] < 25 else '  '
    print(f"║  {r['label']:<30} {params:>7} {r['best_ppl']:>8.1f}{pass_marker} {r['best_step']:>10} {flags:<15} ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  Flags: R=RoPE  N=RMSNorm  S=SwiGLU  Q=QK-Norm  ✓=passes PPL<25   ║")
print("╚══════════════════════════════════════════════════════════════════════╝")

# ── Figure 1: PPL bar chart (all tasks) ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('COMP4680/8650 — NanoGPT ROCStories Results Summary',
             fontsize=13, fontweight='bold')

ax = axes[0]
valid = [r for r in results if r['best_ppl'] is not None]
labels = [r['label'].replace('T2-', '').replace(': ', '\n') for r in valid]
ppls   = [r['best_ppl'] for r in valid]
colors = [r['color'] for r in valid]
bars = ax.bar(range(len(valid)), ppls, color=colors, width=0.65, edgecolor='white', linewidth=0.5)
ax.axhline(25, color='#E24B4A', linestyle='--', linewidth=1.5, label='PPL=25 (pass line)')
for i, (bar, ppl) in enumerate(zip(bars, ppls)):
    ax.text(i, ppl + 0.3, f'{ppl:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(range(len(valid)))
ax.set_xticklabels(labels, fontsize=8, rotation=30, ha='right')
ax.set_ylabel('Best Validation PPL (lower = better)')
ax.set_title('Best val PPL per model (ckpt_best.pt)')
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')

ax2 = axes[1]
curve_configs = [
    ('out-t1-baseline',   'T1 Baseline',   '#888780'),
    ('out-t2-qknorm',     'T2-D +QK-Norm', '#BA7517'),
    ('out-t3-best',       'T3 Best',       '#D85A30'),
    ('out-t4-arena',      'T4 Arena',      '#1D9E75'),
]
for out_dir, label, color in curve_configs:
    log = os.path.join(out_dir, 'train_log.jsonl')
    if not os.path.exists(log): continue
    entries = [json.loads(l) for l in open(log) if 'val_loss' in l]
    if not entries: continue
    steps = [e['step'] for e in entries]
    ppls_c = [math.exp(e['val_loss']) for e in entries]
    ax2.plot(steps, ppls_c, label=label, color=color, linewidth=2)
    best_idx = ppls_c.index(min(ppls_c))
    ax2.scatter([steps[best_idx]], [ppls_c[best_idx]], color=color, s=50, zorder=5)
ax2.axhline(25, color='#E24B4A', linestyle='--', linewidth=1.5, alpha=0.8, label='PPL=25 pass line')
ax2.set_xlabel('Training step'); ax2.set_ylabel('Val PPL')
ax2.set_title('Learning curves — key models')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('final_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Architecture ablation detailed analysis ─────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 4))
fig2.suptitle('Task 2 — Architecture Ablation Analysis', fontsize=12, fontweight='bold')

abl_configs = [
    ('out-t2-vanilla',    'A. Vanilla',           '#888780'),
    ('out-t2-rope',       'B. +RoPE',              '#378ADD'),
    ('out-t2-ffn',        'C. +RMSNorm\n+SwiGLU', '#1D9E75'),
    ('out-t2-qknorm',     'D. +QK-Norm ★',        '#BA7517'),
    ('out-t2-all-modern', 'E. All Modern',         '#7F77DD'),
]

ax3 = axes2[0]
for out_dir, label, color in abl_configs:
    log = os.path.join(out_dir, 'train_log.jsonl')
    if not os.path.exists(log): continue
    entries = [json.loads(l) for l in open(log) if 'val_loss' in l]
    if not entries: continue
    steps = [e['step'] for e in entries]
    val_losses = [e['val_loss'] for e in entries]
    ax3.plot(steps, val_losses, label=label, color=color, linewidth=1.8)
ax3.set_xlabel('Step'); ax3.set_ylabel('Val loss')
ax3.set_title('Validation loss curves')
ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

ax4 = axes2[1]
abl_labels, abl_ppls, abl_colors = [], [], []
for out_dir, label, color in abl_configs:
    log = os.path.join(out_dir, 'train_log.jsonl')
    if not os.path.exists(log): continue
    entries = [json.loads(l) for l in open(log) if 'val_loss' in l]
    if not entries: continue
    best_vl = min(entries, key=lambda e: e['val_loss'])['val_loss']
    abl_labels.append(label); abl_ppls.append(math.exp(best_vl)); abl_colors.append(color)

if abl_ppls:
    bars = ax4.bar(range(len(abl_labels)), abl_ppls, color=abl_colors, width=0.6)
    baseline_ppl = abl_ppls[0] if abl_ppls else 30
    ax4.axhline(baseline_ppl, color='#888780', linestyle='--', alpha=0.6, label='Vanilla baseline')
    for i, (bar, ppl) in enumerate(zip(bars, abl_ppls)):
        ax4.text(i, ppl + 0.1, f'{ppl:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax4.set_xticks(range(len(abl_labels)))
ax4.set_xticklabels(abl_labels, fontsize=8, rotation=20, ha='right')
ax4.set_ylabel('Best Val PPL')
ax4.set_title('Best PPL by configuration\n(D = novel QK-Norm contribution)')
ax4.legend(fontsize=9); ax4.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ablation_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: final_summary.png, ablation_summary.png")

# ── Markdown-ready results for report ─────────────────────────────────────
print("\n── Copy-paste results for Task 2 report ─────────────────────────────")
print("| Config | Architecture | Best Val PPL | Best Step | vs Vanilla |")
print("|--------|-------------|-------------|-----------|------------|")
baseline_ppl_v = next((r['best_ppl'] for r in results if 'T2-A' in r['label']), None)
for r in results:
    if not r['label'].startswith('T2'): continue
    if r['best_ppl'] is None: continue
    delta = r['best_ppl'] - baseline_ppl_v if baseline_ppl_v else 0
    delta_str = f"{delta:+.1f}" if delta != 0 else "—"
    novel = " ★" if 'QK-Norm' in r['label'] else ""
    print(f"| {r['label'].split(': ')[1]+novel:<25} | "
          f"RoPE={'✓' if r['rope'] else '✗'} RMSNorm={'✓' if r['rmsnorm'] else '✗'} "
          f"SwiGLU={'✓' if r['swiglu'] else '✗'} QKNorm={'✓' if r['qknorm'] else '✗'} | "
          f"**{r['best_ppl']:.1f}** | {r['best_step']} | {delta_str} |")

<IPython.core.display.Javascript object>

number of parameters: 29.94M
number of parameters: 29.94M
number of parameters: 29.94M
number of parameters: 29.94M
number of parameters: 29.94M
number of parameters: 29.94M
number of parameters: 29.94M
╔══════════════════════════════════════════════════════════════════════╗
║              FINAL RESULTS SUMMARY — All Tasks                      ║
╠══════════════════════════════════════════════════════════════════════╣
║  Model                           Params  Best PPL  Best Step Flags           ║
╠══════════════════════════════════════════════════════════════════════╣
║  T1: Baseline                       30M     30.0         2250 vanilla         ║
║  T2-A: Vanilla                      30M     30.1         2250 vanilla         ║
║  T2-B: +RoPE                        30M     29.1         1750 R               ║
║  T2-C: +RMSNorm+SwiGLU              30M     30.4         2500 N+S             ║
║  T2-D: +QK-Norm ★                   30M     29.3         2250 Q               ║
║  T2-E: All Mo